# Demo — Benchmarking de Financiación de Automóviles (Honda Bank TFM)

Sube el CSV de ofertas extraído por el scraper y la app calcula automáticamente
los insights clave del análisis competitivo: tipos de interés por marca, efecto
de la electrificación, plazos/valor residual, entidades financiadoras y
clasificación del mercado por nivel de TIN.

Pensado para ejecutarse en Google Colab.

## 1. Instalación de dependencias (solo Colab)

In [1]:
!pip install -q gradio pandas

## 2. Lógica de análisis

In [2]:
import pandas as pd
import gradio as gr

COLUMNAS_NUMERICAS = [
    "precio_vehiculo", "precio_financiar", "promocion_financiacion",
    "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
    "porcentaje_comision_apertura", "comision_apertura",
    "valor_residual", "importe_financiado",
]


def cargar_csv(archivo):
    df = pd.read_csv(archivo.name)
    for col in COLUMNAS_NUMERICAS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def resumen_por_marca(df):
    tabla = (
        df.groupby("marca")
        .agg(
            num_ofertas=("modelo", "count"),
            tin_medio=("tin", "mean"),
            tae_medio=("tae", "mean"),
            cuota_min=("cuota_mensual", "min"),
            cuota_max=("cuota_mensual", "max"),
            comision_media=("porcentaje_comision_apertura", "mean"),
        )
        .round(2)
        .sort_values("tin_medio")
        .reset_index()
    )
    return tabla


def resumen_por_combustible(df):
    tabla = (
        df.groupby("tipo_combustible")
        .agg(
            num_ofertas=("modelo", "count"),
            tin_medio=("tin", "mean"),
            tae_medio=("tae", "mean"),
            cuota_media=("cuota_mensual", "mean"),
        )
        .round(2)
        .sort_values("tin_medio")
        .reset_index()
    )
    return tabla


def diferencial_electrificacion(df):
    """TIN medio eléctrico/PHEV vs. combustión por marca, y el diferencial en puntos."""
    df = df.copy()
    df["grupo_propulsion"] = df["tipo_combustible"].apply(
        lambda x: "electrificado" if x in ("eléctrico", "híbrido enchufable") else "combustión/híbrido"
    )
    pivot = (
        df.groupby(["marca", "grupo_propulsion"])["tin"]
        .mean()
        .unstack()
        .round(2)
    )
    if "electrificado" in pivot.columns and "combustión/híbrido" in pivot.columns:
        pivot["diferencial_pp"] = (pivot["electrificado"] - pivot["combustión/híbrido"]).round(2)
    pivot = pivot.sort_values("diferencial_pp") if "diferencial_pp" in pivot.columns else pivot
    return pivot.reset_index()


def plazos_y_residual(df):
    tabla = (
        df.groupby("marca")
        .agg(
            plazo_habitual=("plazo_meses", lambda s: s.mode().iloc[0] if not s.mode().empty else None),
            valor_residual_medio=("valor_residual", "mean"),
            entrada_media=("entrada", "mean"),
        )
        .round(0)
        .reset_index()
    )
    return tabla


def entidades_financiadoras(df):
    tabla = (
        df.groupby("marca")["banco_financiacion"]
        .agg(lambda s: ", ".join(sorted(s.dropna().unique())))
        .reset_index()
        .rename(columns={"banco_financiacion": "entidad(es)"})
    )
    return tabla


def clasificacion_mercado(df):
    medias = df.groupby("marca")["tin"].mean().round(2)

    def nivel(tin):
        if tin < 6:
            return "Bajo (<6%)"
        elif tin < 7.5:
            return "Medio (6–7.5%)"
        else:
            return "Alto (>7.5%)"

    out = medias.reset_index()
    out["nivel_tin"] = out["tin"].apply(nivel)
    out = out.sort_values("tin").rename(columns={"tin": "tin_medio"})
    return out


def generar_texto_resumen(df):
    n_ofertas = len(df)
    n_marcas = df["marca"].nunique()
    pct_pcp = (df["tipo_financiacion"] == "PCP").mean() * 100

    marca_tin_min = df.groupby("marca")["tin"].mean().idxmin()
    tin_min = df.groupby("marca")["tin"].mean().min()
    marca_tin_max = df.groupby("marca")["tin"].mean().idxmax()
    tin_max = df.groupby("marca")["tin"].mean().max()

    fuel_tin = df.groupby("tipo_combustible")["tin"].mean()
    combustible_mas_barato = fuel_tin.idxmin()
    combustible_mas_caro = fuel_tin.idxmax()

    texto = f"""### Resumen automático

- **{n_ofertas} ofertas** analizadas de **{n_marcas} marcas**. El **{pct_pcp:.0f}%** se financia bajo la modalidad PCP.
- TIN medio más bajo: **{marca_tin_min}** ({tin_min:.2f}%). TIN medio más alto: **{marca_tin_max}** ({tin_max:.2f}%).
- Por tipo de propulsión, **{combustible_mas_barato}** tiene el TIN medio más bajo ({fuel_tin.min():.2f}%) y **{combustible_mas_caro}** el más alto ({fuel_tin.max():.2f}%), lo que confirma la estrategia de tipos diferenciados para favorecer la electrificación.
- Recuerda: este análisis es un *diagnóstico del mercado competidor* (snapshot del día de extracción), no una comparación directa con las condiciones de Honda Bank.
"""
    return texto


def procesar(archivo):
    if archivo is None:
        return "Sube primero un CSV.", None, None, None, None, None, None

    df = cargar_csv(archivo)

    texto = generar_texto_resumen(df)
    t_marca = resumen_por_marca(df)
    t_combustible = resumen_por_combustible(df)
    t_electrif = diferencial_electrificacion(df)
    t_plazos = plazos_y_residual(df)
    t_entidades = entidades_financiadoras(df)
    t_clasificacion = clasificacion_mercado(df)

    return texto, t_marca, t_combustible, t_electrif, t_plazos, t_entidades, t_clasificacion


## 3. Interfaz Gradio

In [ ]:
with gr.Blocks(title="Benchmarking Financiación Auto — Honda Bank") as demo:
    gr.Markdown("# Benchmarking de Financiación de Automóviles\nSube el CSV de ofertas de la competencia (salida del scraper) para ver los insights clave.")

    archivo_input = gr.File(label="CSV de ofertas", file_types=[".csv"])
    boton = gr.Button("Analizar", variant="primary")

    resumen_md = gr.Markdown()

    with gr.Tab("Por marca"):
        tabla_marca = gr.Dataframe(label="TIN/TAE/cuota medios por marca (ordenado por TIN)")
    with gr.Tab("Por propulsión"):
        tabla_combustible = gr.Dataframe(label="TIN/TAE/cuota medios por tipo de combustible")
    with gr.Tab("Efecto electrificación"):
        tabla_electrif = gr.Dataframe(label="TIN medio: electrificado vs. combustión/híbrido, por marca")
    with gr.Tab("Plazos y valor residual"):
        tabla_plazos = gr.Dataframe(label="Plazo habitual, valor residual y entrada medios por marca")
    with gr.Tab("Entidades financiadoras"):
        tabla_entidades = gr.Dataframe(label="Banco(s) que financia(n) cada marca")
    with gr.Tab("Clasificación del mercado"):
        tabla_clasificacion = gr.Dataframe(label="Marcas agrupadas por nivel de TIN (bajo/medio/alto)")

    boton.click(
        procesar,
        inputs=[archivo_input],
        outputs=[resumen_md, tabla_marca, tabla_combustible, tabla_electrif, tabla_plazos, tabla_entidades, tabla_clasificacion],
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/06/30 20:18:52 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>